# Hyperliquid consolidated account analytics

Combines balances, account history, fills, funding, positions, and PnL across the configured Hyperliquid wallets. Combined account-value history is shown only over the dates available for every wallet; charts show the oldest wallet's account history before shared coverage begins, then the combined account value.

Daily rollups and calendar-period cutoffs use `America/New_York`. Previously saved UTC-aggregated daily totals are mapped to Eastern date labels when loaded; their original intraday split cannot be reconstructed.


## Wallets and implementation

Edit `WALLET_ADDRESSES` here. Both addresses found in the single-wallet notebook are included: the active address and the previously commented address.


In [41]:
import json
from pathlib import Path

WALLET_ADDRESSES = [
    '0xf3fb3598bFfA608d1e7CF2837d74F02B1191c248',
    '0x7532dA34edA8f3e383bC480d50e7A7d3b12c07f2',
]

RISK_BUDGET_PCT = 5.0
ATR_PERIOD = 14
ATR_STOP_MULTIPLE = 1.25
MARGIN_RESERVE_PCT = 20.0
namespace={'__name__':'hyperliquid_multi_account_analytics'}
implementation_paths=[
    Path('hyperliquid_account_analytics_implementation.ipynb'),
    Path('hyperliquid_multi_account_analytics_implementation.ipynb'),
]
for implementation_path in implementation_paths:
    implementation=json.loads(implementation_path.read_text(encoding='utf-8'))
    for cell in implementation['cells']:
        if cell['cell_type']=='code':
            source=''.join(cell['source'])
            if source.strip():
                exec(compile(source,f'{implementation_path} cell','exec'),namespace)
MultiWalletAnalytics=namespace['MultiWalletAnalytics']
analytics=MultiWalletAnalytics(WALLET_ADDRESSES,data_dir='data')
analytics.position_risk_budget_pct=RISK_BUDGET_PCT
analytics.position_atr_period=ATR_PERIOD
analytics.position_atr_stop_multiple=ATR_STOP_MULTIPLE
analytics.margin_reserve_pct=MARGIN_RESERVE_PCT


## Fetch and consolidate wallet data

Fetch each wallet once and prepare combined account history and fills.


In [42]:
%%capture
_ = analytics.fetch_data()


## Account / PnL

Combined account and risk summary. Daily PnL and calendar-period cutoffs use US Eastern time.


In [43]:
analytics.show_account_pnl()


=== ACCOUNT / PNL AND RISK ===


Metric,Value
Account value,"$2,918.78"
Portfolio value,"$3,812.22 (combined live wallet value)"
Account 1D (ET),2.39%
Realized today (ET),$-0.29
Realized today (% of account value),-0.01%
Unrealized PnL,$5.42
Unrealized PnL (% of account value),0.19%
Unrealized 1D change (ET),n/a
Average realized PnL / day,$-0.96
Average unrealized PnL change / day,n/a


## Account charts

Account value and PnL charts. Before all wallets have saved history, account value and weekly returns use the oldest wallet alone; from shared coverage onward they use the combined accounts.


In [44]:
analytics.show_account_charts()


Weekly return chart omits 2 return(s) below -100%;                     other data and stats are unchanged.


## Account vs BTC

Daily account-vs-BTC comparison uses the oldest wallet before shared history begins, then the combined accounts. The lifetime realized-PnL comparison pools the available wallet history.


In [45]:
analytics.show_account_vs_btc()


Lifetime account series is anchored to current account value; it includes fee-net realized trades and perp funding, but excludes historical cash flows and unrealized PnL.


## Traded volume vs realized PnL chart

Daily fill volume and realized PnL pooled across wallets.


In [46]:
analytics.show_volume_chart()


## Open positions

Show each wallet position separately, followed by fixed-fractional sizing grouped by asset and side and an aggregate summary. The summary includes cumulative open-position funding as a percentage of combined account equity and total position value. Reserve-adjusted available margin is combined account equity less the configured reserve; the summary shows that capacity, margin used, remaining available margin in dollars and as a percentage of that capacity, and position value as a percentage of equity. Set `MARGIN_RESERVE_PCT` (20% by default) in the setup cell to change the reserve. The position table includes funding and fees since open as dollar amounts, plus fees since open as percentages of wallet equity and position value and position value as a percentage of wallet account value. Fee ratios are available for complete USDC-denominated fee history. Set `RISK_BUDGET_PCT` (5% per asset), `ATR_PERIOD` (14 completed daily candles), and `ATR_STOP_MULTIPLE` (1.25) in the setup cell. Stops are placed 1.25 ATR from the current mark, and take-profit is set the same distance in the favorable direction (1:1 risk/reward). Same-side positions across wallets are combined; long and short rows remain separate. If both sides are open for one asset, they share that asset's risk budget equally.


In [47]:
analytics.show_open_positions()


Wallet,Coin,Dex,Side,Liq %,ROE,uPnL,Funding (since open),Funding (1h),Funding APR,Expected Price PnL (5d),Expected Funding PnL (5d),Expected PnL incl. funding (5d),Entry,Mark,Pos,Margin,Margin Type,Margin %,Fees,Fees / Equity (%),Fees / Position (%),Position / Account (%),Liq Px,Lev.,Sz
c248,IMX,,SHORT,62.33%,13.89%,$27.55,$0.63,0.001250%,10.95%,$12.16,$1.44,$13.61,0.12332,0.11985,$964,$193,Cross,25.99%,-$0.14,0.013%,0.015%,86.50%,0.19455,5.00x,"-8,040.50"
07f2,TRUMP,,SHORT,187.99%,30.23%,$57.88,$0.03,0.001250%,10.95%,$11.07,$0.28,$11.36,1.75136,1.6192,$708,$177,Cross,14.48%,-$0.20,0.007%,0.028%,26.24%,4.6631,4.00x,-437.30
07f2,TRX,,SHORT,188.70%,-12.52%,-$8.73,$0.06,0.001250%,10.95%,-$1.45,-$1.26,-$2.70,0.313,0.31692,$705,$71,Cross,5.77%,-$0.10,0.004%,0.014%,26.15%,0.91494,10.00x,"-2,226.00"
07f2,xyz:GOLD,xyz,LONG,n/a,-1.20%,-$1.08,-$0.16,0.002883%,25.25%,-$0.99,-$0.66,-$1.65,"3,909.8","3,903.2",$630,$90,Cross,7.36%,-$0.18,0.007%,0.029%,23.34%,0,7.00x,0.16
07f2,PENGU,,SHORT,205.07%,10.93%,$13.84,$0.30,0.001250%,10.95%,$2.08,$0.54,$2.62,0.007292,0.007135,$619,$124,Cross,10.13%,-$0.09,0.003%,0.015%,22.96%,0.021767,5.00x,"-86,832.00"
07f2,xyz:GOOGL,xyz,LONG,n/a,-53.19%,-$103.21,-$2.40,0.002101%,18.40%,$0.12,-$0.66,-$0.54,343.02,282.21,$479,$160,Cross,13.06%,-$0.05,0.002%,0.011%,17.75%,0,3.00x,1.70
07f2,MELANIA,,SHORT,279.83%,5.46%,$7.94,$0.08,0.001250%,10.95%,-$0.44,$0.24,-$0.20,0.07686,0.07545,$428,$143,Cross,11.68%,-$0.06,0.002%,0.015%,15.87%,0.28658,3.00x,"-5,674.80"
07f2,io:PRL,io,SHORT,11.04%,-11.94%,-$16.31,$6.17,0.017098%,149.78%,$19.16,$11.92,$31.08,1.50057,1.5603,$426,$126,Isolated,10.30%,-$0.04,0.001%,0.009%,15.79%,1.7325,3.38x,-273.00
07f2,xyz:JIO,xyz,LONG,30.73%,-4.23%,-$4.69,$5.99,-0.004826%,-42.28%,$25.61,$11.80,$37.41,14.54696,14.342,$328,$112,Isolated,9.19%,-$0.03,0.001%,0.009%,12.17%,9.9343,2.92x,22.90
c248,USELESS,,SHORT,182.38%,17.91%,$19.70,$0.27,0.001250%,10.95%,$12.54,$0.56,$13.10,0.132058,0.12414,$310,$103,Cross,13.95%,-$0.09,0.008%,0.028%,27.86%,0.35055,3.00x,"-2,500.00"


Funding and fees are cumulative since the position was opened. Fee ratios are shown only for complete USDC-denominated fees; otherwise they are n/a.
=== ACCOUNT NET EXPOSURE AND FUNDING ===


Net Direction,Long Positions,Short Positions,Long Notional ($),Short Notional ($),Gross Notional ($),Net Notional ($),Net Exposure / Gross (%),Net Funding Rate / 1h (%) (covered),"Net Funding APR (%) (simple, covered)",Estimated Net Funding PnL / hour ($),Estimated Net Funding PnL / day ($),Funding-Covered Notional ($),Funding Coverage (%)
NET SHORT,6,16,"$1,753.20","$5,696.16","$7,449.36","$-3,942.96",-52.93%,0.001684%,14.75%,$0.1255,$3.01,"$7,449.36",100.0%


Net direction compares long and short USD notionals; it is not delta- or beta-adjusted. Net funding rate is estimated net funding cash flow divided by funding-covered gross notional. Positive cash flow means estimated receipts. APR assumes current hourly rates persist; it is not a forecast.
=== FUNDING BY ASSET AND SIDE ===


Coin,Side,Positions,Size,Position Value ($),Current Funding / 1h (%),Funding APR (%),Estimated Funding PnL / hour ($),Estimated Funding PnL / day ($),Funding Coverage (%)
ADA,SHORT,1,"1,086",$206.52,0.000912%,7.99%,$0.0019,$0.05,100.0%
AERO,LONG,1,90,$55.79,0.001149%,10.06%,$-0.0006,$-0.02,100.0%
AVAX,SHORT,1,30,$257.83,0.001250%,10.95%,$0.0032,$0.08,100.0%
FOGO,SHORT,1,7.942e+04,$203.39,0.001250%,10.95%,$0.0025,$0.06,100.0%
GRASS,SHORT,1,452.3,$258.69,0.001250%,10.95%,$0.0032,$0.08,100.0%
IMX,SHORT,1,"8,040",$964.06,0.001250%,10.95%,$0.0121,$0.29,100.0%
INIT,SHORT,1,"1,105",$170.79,0.001250%,10.95%,$0.0021,$0.05,100.0%
LIT,SHORT,1,9,$27.82,-0.002522%,-22.10%,$-0.0007,$-0.02,100.0%
MELANIA,SHORT,1,"5,675",$428.22,0.001250%,10.95%,$0.0054,$0.13,100.0%
PENGU,SHORT,1,8.683e+04,$619.37,0.001250%,10.95%,$0.0077,$0.19,100.0%


Funding rates are current hourly rates, weighted by position notional across wallets. Positive rates are paid by longs and received by shorts; estimated cash flows assume the current rate stays constant. APR is simple annualization, not a forecast.
Fixed-fractional sizing: risk budget 5.00% of combined account value per asset; target quantity uses 1.25 x daily ATR(14) plus the directionally adverse 95th-percentile five-day funding reserve. Opposing sides, when both open, split the asset budget equally.
Limited daily history; ATR uses all completed candles available (shorter than 14 days) for: xyz:JIO.
ATR stop and take-profit distances currently both use 1.25 x daily ATR from the current mark (1:1 price reward/risk before funding, fees, and slippage).
ATR stop-distance colors: green <2%, orange 2-5%, red >5%; red mark means the position is out of the money; stop prices are highlighted red. Open-risk includes the funding reserve but excludes fees, slippage, and gaps; the stop trigger it

Asset,Side,ATR (%),ATR Period Used,ATR Stop Distance (%),ATR Take Profit Distance (%),Mark to Take Profit (%),Funding Reserve / unit,Avg Entry,Mark,Stop,Take Profit,Open - Target (coins),Open - Target (%),Open Risk (price only) ($),Open Risk incl. funding ($)
ADA,SHORT,6.36%,14 days,7.95%,7.95%,7.95%,$0.000350222,0.193547,0.19021,0.20532,0.1751,"11,240.8",-91.19%,$16.41,$16.79
AERO,LONG,9.98%,14 days,12.48%,12.48%,12.48%,$0.000902483,0.832395,0.61988,0.54252,0.69724,"-2,345.47",-96.30%,$6.96,$7.04
AVAX,SHORT,6.38%,14 days,7.98%,7.98%,7.98%,$0.00298656,8.7597,8.594,9.2796,7.9084,246.831,-89.16%,$20.57,$20.66
FOGO,SHORT,14.50%,14 days,18.13%,18.13%,18.13%,$0,0.002597,0.002561,0.0030253,0.0020967,"331,089",-80.65%,$36.88,$36.88
GRASS,SHORT,13.46%,14 days,16.83%,16.83%,16.83%,$0,0.60132,0.57194,0.66819,0.47569,"1,527.99",-77.16%,$43.54,$43.54
IMX,SHORT,8.96%,14 days,11.21%,11.21%,11.21%,$0,0.12332,0.11985,0.13328,0.10642,"6,152.85",-43.35%,$107.98,$107.98
INIT,SHORT,17.54%,14 days,21.93%,21.93%,21.93%,$0.000698722,0.18359,0.15495,0.18893,0.12097,"4,391.86",-79.90%,$37.55,$38.32
LIT,SHORT,11.77%,14 days,14.71%,14.71%,14.71%,$0.00350161,3.2512,3.0904,3.545,2.6358,407.046,-97.84%,$4.09,$4.12
MELANIA,SHORT,7.61%,14 days,9.52%,9.52%,9.52%,$5.63973e-05,0.07686,0.07545,0.082631,0.068269,"20,661",-78.45%,$40.75,$41.07
PENGU,SHORT,8.24%,14 days,10.30%,10.30%,10.30%,$8.79063e-06,0.007292,0.007135,0.0078698,0.0064002,"169,509",-66.13%,$63.80,$64.57


Metric,Value
Wallets,2
Position value,"$7,449"
uPnL,$5.42
ROE,0.29%
Margin,"$1,878"
Margin % bal,49.33%
Available margin (80% of equity),"$3,049.78"
Available margin used (20% reserve),61.58%
Available margin remaining,"$1,171.76"
Available margin remaining (%),38.42%


ROE is unrealized PnL divided by used margin; a negative ROE means the open positions are currently at an unrealized loss relative to the margin supporting them.


## Realized PnL statistics

Pool spot/perp fills and signed funding across wallets.


In [48]:
analytics.show_realized_pnl()


=== REALIZED SPOT ===


Metric,Value
Realized PnL,$22.35 (0.59%)
24H,$0.00 (0.00%)
7D,$0.00 (0.00%)
30D,$22.35 (0.59%)
QTD,$22.35 (0.59%)
YTD,$22.35 (0.59%)
W,3
Win %,100.00%
Avg W,$7.45
L,0


=== REALIZED PERP ===


Metric,Value
Realized PnL,$-65.44 (-1.72%)
24H,$14.62 (0.38%)
7D,$353.05 (9.26%)
30D,$458.62 (12.03%)
QTD,$458.48 (12.03%)
YTD,$458.35 (12.02%)
W,190
Win %,67.62%
Avg W,$8.37
L,91



=== PERP PNL BREAKDOWN ===


Component,Amount ($),Contribution (% of |net PnL|)
Trading,$87.07,133.05%
Fees,$-107.21,-163.82%
Funding,$-45.31,-69.23%
Net perp PnL,$-65.44,


Liquidations: 0 of 281 closes (0.00%) | liquidation PnL: $0.00.


## Traded volume vs PnL

Combined turnover efficiency and correlations.


In [49]:
analytics.show_volume_summary()


=== TRADED VOLUME VS REALIZED PNL (ALL WALLETS) ===


Market,Volume ($),Realized PnL ($),Realized PnL / Volume (%),PnL / $1k volume ($),Active days,Daily Pearson r,Daily Spearman r
Perps,"$260,593.34",$-20.14,-0.01%,$-0.08,58,0.102,0.267
Spot,"$1,397.53",$22.35,1.60%,$16.00,5,-0.509,-0.462


## PnL attribution

Combined realized and open-position PnL by asset.


In [50]:
analytics.show_pnl_attribution()


=== CONSOLIDATED PNL ATTRIBUTION ===


Type,Asset,Closes,Realized,Realized (% gross),uPnL,uPnL (% gross),Total PnL,Total (% gross)
Perp,BTC,12,$-475.21,-18.90%,$0.00,0.00%,$-475.21,-16.73%
Perp,kBONK,3,$-397.06,-15.79%,$0.00,0.00%,$-397.06,-13.98%
Perp,ZORA,21,$304.49,12.11%,$0.00,0.00%,$304.49,10.72%
Perp,KAITO,4,$-193.84,-7.71%,$0.00,0.00%,$-193.84,-6.82%
Perp,HYPE,11,$146.54,5.83%,$0.00,0.00%,$146.54,5.16%
Perp,VVV,40,$116.26,4.62%,$0.00,0.00%,$116.26,4.09%
Perp,ETH,15,$114.54,4.56%,$0.00,0.00%,$114.54,4.03%
Perp,xyz:GOOGL,0,$-2.73,-0.11%,$-103.21,-27.84%,$-105.94,-3.73%
Perp,io:ANTH,2,$-86.69,-3.45%,$0.00,0.00%,$-86.69,-3.05%
Perp,TRUMP,0,$0.03,0.00%,$60.77,16.39%,$60.81,2.14%


Net attributed PnL across wallets: $-37.61.
